In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from pathlib import Path
import logging
import pandas as pd
from datasets import Dataset
from flwr_datasets.partitioner import IidPartitioner
from datasets import Dataset, Sequence, Value, Features

/home/alf/.pyenv/versions/tfg_flwr/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Nota:** no muevas el script de sitio o se rompe

# Preprocesado de datos

Este script nos va a permitir preparar los datasets para el entrenamiento del modelo, esto lo vamos a realizar de dos maneras diferentes: normalización y expresión en forma de línea temporal a través de secuencias.

La primero vamos a realizar una normalización de los datos y los colocaremos de tal manera que cada uno de los que los datos de entrada sean las lecturas de los sensores en las últimas 10 horas y la salida será la temperatura que va ha hacer en la siguiente hora. Esto require que el dataset sea de dos dimensiones, así que se guardará como JSON.

NOTA: aquí vamos a desglosar y adaptar la función para pleidata de la clase `DataLoader` original. Cambiando también la configuración por constantes.

In [2]:
def create_sequences(features, target, window_size):
    """Función base unificada para crear secuencias."""
    X_list, Y_list = [], []
    for i in range(len(features) - window_size):
        seq_x = features[i: i + window_size]
        seq_y = target[i + window_size]
        X_list.append(seq_x)
        Y_list.append(seq_y)
    return np.array(X_list), np.array(Y_list)

In [ ]:
# 1. Definir Configuración
feature_cols = ['dif_cons_real', 
        'dif_cons_smooth', 
        'V2', 
        'V4', 
        'V12', 
        'V26', 
        'Hour_1', 
        'Hour_2', 
        'Hour_3', 
        'Season_1', 
        'Season_2', 
        'Season_3', 
        'Season_4', 
        'tmed', 
        'hrmed', 
        'radmed', 
        'vvmed', 
        'dvmed', 
        'prec', 
        'dewpt', 
        'dpv']
target_col: str = 'dif_cons_real'
window_size:int = 12
dataset:str = 'data-model-consumoA-60T.csv'
output_dir_name: str = 'buildingA-data'

In [28]:
# 2. Cargar Datos
base_dir = Path('..')
file_path =  base_dir / 'og' / dataset # NOTE: Si has movido el script arregla aquí
if not file_path.exists():
    raise FileNotFoundError(f"Fichero de datos no encontrado en {file_path}")

df = pd.read_csv(file_path, sep=';', index_col=0)

In [29]:
# 3. Limpieza (sin escalar aquí)
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df.dropna(subset=feature_cols, inplace=True)

if len(df) <= window_size:
    raise ValueError(f"Datos insuficientes tras limpieza ({len(df)} filas) para window_size ({window_size})")

In [30]:
# 4. Escalado (UNA SOLA VEZ)
scaler = MinMaxScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

In [ ]:
# 5. Crear Secuencias
features = df[feature_cols].values.astype(np.float32)
target = df[target_col].values.astype(np.float32)
X_seq, y_seq = create_sequences(features, target, window_size)

In [33]:
data = {
    "features": X_seq.tolist(),  # lista de lista de listas: [[[...], [...], [...]], ...]
    "label":    y_seq.tolist()
}

features_schema = Features({
    "features": Sequence(Sequence(Value("float64"), length=len(feature_cols)), length=window_size),
    "label":    Value("int64")
})

dataset = Dataset.from_dict(data, features=features_schema)

# Split into train (80%) and test (20%)
dataset_split = dataset.train_test_split(test_size=0.2, seed=42)

print("Dataset split:")
print(f"  Train samples: {len(dataset_split['train'])}")
print(f"  Test samples: {len(dataset_split['test'])}")
print(dataset_split['train'].features)
print(dataset_split['test'].features)

Dataset split:
  Train samples: 6603
  Test samples: 1651
{'features': List(List(Value('float64'), length=21), length=12), 'label': Value('int64')}
{'features': List(List(Value('float64'), length=21), length=12), 'label': Value('int64')}


In [34]:
# Export to JSON as DatasetDict with train/test split
import json
import os

# Get the base name without extension
base_output_name = output_dir_name.replace('.parquet', '').replace('.json', '')

# Create a directory for this dataset
dataset_dir = f"../{base_output_name}"
os.makedirs(dataset_dir, exist_ok=True)

# Save both train and test splits
dataset_split['train'].to_json(f"{dataset_dir}/train.json")
dataset_split['test'].to_json(f"{dataset_dir}/test.json")

print(f"Dataset saved to {dataset_dir}/")
print(f"  - {dataset_dir}/train.json")
print(f"  - {dataset_dir}/test.json")

Creating json from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 48.99ba/s]

Dataset saved to ../buildingB-data/
  - ../buildingB-data/train.json
  - ../buildingB-data/test.json


# Test

Vamos a importar el fichero para comprobar si se ha exportado correctamente

In [35]:
# Cargando datos de fichero (pre-split into train/test)
train_dataset = Dataset.from_json(f"../{output_dir_name}/train.json")
test_dataset = Dataset.from_json(f"../{output_dir_name}/test.json")

# Create a DatasetDict
from datasets import DatasetDict
dataset_loaded = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

print(f"Dataset loaded: {dataset_loaded}")
print(f"Train samples: {len(dataset_loaded['train'])}")
print(f"Test samples: {len(dataset_loaded['test'])}")

NUM_CLIENTS = 5 # Imagina que tienes cinco vacas clientes
partitioner = IidPartitioner(num_partitions=NUM_CLIENTS)
partitioner.dataset = dataset_loaded["train"]

# Simulate client 0 training
partition = partitioner.load_partition(partition_id=0)
partition_np = partition.with_format("numpy")

X_client = np.stack([partition_np["features"]], axis=1)
y_client = np.stack([partition_np["label"]], axis = 1)

print(f"Client 0 -> X shape: {X_client.shape}, y shape: {y_client.shape}")

Generating train split: 6603 examples [00:00, 46490.57 examples/s]
Generating train split: 1651 examples [00:00, 46839.48 examples/s]


Dataset loaded: DatasetDict({
    train: Dataset({
        features: ['features', 'label'],
        num_rows: 6603
    })
    test: Dataset({
        features: ['features', 'label'],
        num_rows: 1651
    })
})
Train samples: 6603
Test samples: 1651
Client 0 -> X shape: (1321, 1, 12, 21), y shape: (1321, 1)


In [5]:
import os

DATASETS_FOLDER = "../datasets"
if not os.path.isdir(DATASETS_FOLDER):
    print("La carpeta de datasets no existe.")


datasets = [str(os_file) for os_file in os.listdir(DATASETS_FOLDER)]
datasets 


['buildingB-data', 'buildingA-data', 'buildingC-data']

In [ ]:

train_dataset = Dataset.from_json(f"../datasets/{datasets[0]}/train.json")
test_dataset = Dataset.from_json(f"../datasets/{datasets[0]}/test.json")

# Create a DatasetDict
from datasets import DatasetDict
dataset_loaded = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

print(f"Dataset loaded: {dataset_loaded}")
print(f"Train samples: {len(dataset_loaded['train'])}")
print(f"Test samples: {len(dataset_loaded['test'])}")

NUM_CLIENTS = 5 # Imagina que tienes cinco vacas clientes
partitioner = IidPartitioner(num_partitions=NUM_CLIENTS)
partitioner.dataset = dataset_loaded["train"]

# Simulate client 0 training
partition = partitioner.load_partition(partition_id=0)
partition_np = partition.with_format("numpy")

X_client = np.stack([partition_np["features"]], axis=1)
y_client = np.stack([partition_np["label"]], axis = 1)

print(f"Client 0 -> X shape: {X_client.shape}, y shape: {y_client.shape}")

FileNotFoundError: Unable to find '/home/alf/Documents/rewrite_tfg/data/scripts/../buildingB-data/train.json'